# Heart Attack Risk Dataset (China): Data Study and Simple EDA

## 1. Loading the data

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from scipy.stats import gaussian_kde   # only used to compute the KDE curve

In [2]:
df = pd.read_csv("heart_attack_china.csv")
df.head()

,Patient_ID,Age,Gender,Smoking_Status,Hypertension,Diabetes,Obesity,Cholesterol_Level,Air_Pollution_Exposure,Physical_Activity,...,Hospital_Availability,TCM_Use,Employment_Status,Education_Level,Income_Level,Blood_Pressure,Chronic_Kidney_Disease,Previous_Heart_Attack,CVD_Risk_Score,Heart_Attack
0,1,55,Male,Non-Smoker,No,No,Yes,Normal,High,High,...,Low,Yes,Unemployed,Primary,Low,104,Yes,No,78,No
1,2,66,Female,Smoker,Yes,No,No,Low,Medium,High,...,High,No,Unemployed,Secondary,Middle,142,No,No,49,No
2,3,69,Female,Smoker,No,No,No,Low,Medium,High,...,Low,No,Unemployed,Primary,High,176,No,No,31,No
3,4,45,Female,Smoker,No,Yes,No,Normal,Medium,Low,...,Medium,Yes,Employed,Primary,Low,178,No,Yes,23,No
4,5,39,Female,Smoker,No,No,No,Normal,Medium,Medium,...,Low,No,Retired,Higher,Middle,146,Yes,No,79,No


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 239266 entries, 0 to 239265
Data columns (total 28 columns):
 #   Column                  Non-Null Count   Dtype
---  ------                  --------------   -----
 0   Patient_ID              239266 non-null  int64
 1   Age                     239266 non-null  int64
 2   Gender                  239266 non-null  str  
 3   Smoking_Status          239266 non-null  str  
 4   Hypertension            239266 non-null  str  
 5   Diabetes                239266 non-null  str  
 6   Obesity                 239266 non-null  str  
 7   Cholesterol_Level       239266 non-null  str  
 8   Air_Pollution_Exposure  239266 non-null  str  
 9   Physical_Activity       239266 non-null  str  
 10  Diet_Score              239266 non-null  str  
 11  Stress_Level            239266 non-null  str  
 12  Alcohol_Consumption     239266 non-null  str  
 13  Family_History_CVD      239266 non-null  str  
 14  Healthcare_Access       239266 non-null  str  
 15  Rural_or_Ur

In [4]:
df.describe(include="all")

,Patient_ID,Age,Gender,Smoking_Status,Hypertension,Diabetes,Obesity,Cholesterol_Level,Air_Pollution_Exposure,Physical_Activity,...,Hospital_Availability,TCM_Use,Employment_Status,Education_Level,Income_Level,Blood_Pressure,Chronic_Kidney_Disease,Previous_Heart_Attack,CVD_Risk_Score,Heart_Attack
count,239266.000000,239266.000000,239266,239266,239266,239266,239266,239266,239266,239266,...,239266,239266,239266,179649,239266,239266.000000,239266,239266,239266.000000,239266
unique,NaN,NaN,2,2,2,2,2,3,3,3,...,3,2,3,3,3,NaN,2,2,NaN,2
top,NaN,NaN,Female,Non-Smoker,No,No,No,Normal,Medium,Low,...,High,No,Retired,Higher,High,NaN,No,No,NaN,No
freq,NaN,NaN,119948,119697,174557,212794,167521,80091,119593,80086,...,79848,143949,79872,59942,80149,NaN,203597,215306,NaN,210195
mean,119633.500000,59.466284,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,134.513378,NaN,NaN,54.485581,NaN
std,69070.289091,17.348308,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,25.991827,NaN,NaN,25.987743,NaN
min,1.000000,30.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,90.000000,NaN,NaN,10.000000,NaN
25%,59817.250000,44.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,112.000000,NaN,NaN,32.000000,NaN
50%,119633.500000,59.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,135.000000,NaN,NaN,54.000000,NaN
75%,179449.750000,75.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,157.000000,NaN,NaN,77.000000,NaN


In [5]:
df.isnull().sum()

Patient_ID                    0
Age                           0
Gender                        0
Smoking_Status                0
Hypertension                  0
Diabetes                      0
Obesity                       0
Cholesterol_Level             0
Air_Pollution_Exposure        0
Physical_Activity             0
Diet_Score                    0
Stress_Level                  0
Alcohol_Consumption           0
Family_History_CVD            0
Healthcare_Access             0
Rural_or_Urban                0
Region                        0
Province                      0
Hospital_Availability         0
TCM_Use                       0
Employment_Status             0
Education_Level           59617
Income_Level                  0
Blood_Pressure                0
Chronic_Kidney_Disease        0
Previous_Heart_Attack         0
CVD_Risk_Score                0
Heart_Attack                  0
dtype: int64

Only `Education_Level` shows nulls (59,617), and no other column has any. That is odd for a clean dataset, so I looked at the raw values. Pandas treats the text `None` as a missing value by default, so I reloaded the file with `keep_default_na=False` and checked again.

In [6]:
df = pd.read_csv("heart_attack_china.csv", keep_default_na=False)
print("Total null values:", df.isnull().sum().sum())
df["Education_Level"].value_counts()

Total null values: 0


Education_Level
Higher       59942
Primary      59901
Secondary    59806
None         59617
Name: count, dtype: int64

There are no null values in the dataset. `None` in `Education_Level` is not an error or an empty cell, it is one of the four classes (None, Primary, Secondary, Higher) and means the patient has no formal education. So those 59,617 rows stay as they are.

## 2. Data dictionary

In [15]:
GROUPS = {
    "Identifier":        ["Patient_ID"],
    "Demographic":       ["Age", "Gender"],
    "Clinical":          ["Hypertension", "Diabetes", "Obesity", "Cholesterol_Level", "Blood_Pressure",
                          "Chronic_Kidney_Disease", "Previous_Heart_Attack", "Family_History_CVD", 
                          "CVD_Risk_Score"],
    "Lifestyle":         ["Smoking_Status", "Alcohol_Consumption", "Physical_Activity", "Diet_Score", "Stress_Level"],
    "Environment":       ["Air_Pollution_Exposure"],
    "Geography":         ["Region", "Province", "Rural_or_Urban"],
    "Healthcare access": ["Healthcare_Access", "Hospital_Availability"],
    "Socioeconomic":     ["Employment_Status", "Education_Level", "Income_Level", "TCM_Use"],
    "Outcome":           ["Heart_Attack", "Heart_Attack_Flag"],
}
group_of = {col: group for group, cols in GROUPS.items() for col in cols}

rows = []
for col in df.columns:
    s = df[col]
    if pd.api.types.is_numeric_dtype(s):
        kind, values = "numeric", f"{s.min()} to {s.max()}"
    else:
        kind = "categorical"
        vals = s.value_counts().index.tolist()
        values = ", ".join(vals) if len(vals) <= 8 else f"{len(vals)} values (e.g. {', '.join(vals[:4])}, ...)"
    rows.append({"column": col, "group": group_of[col], "type": kind, "distinct": s.nunique(), "values / range": values})

pd.DataFrame(rows).sort_values(["group", "column"]).reset_index(drop=True)

,column,group,type,distinct,values / range
0,Blood_Pressure,Clinical,numeric,90,90 to 179
1,CVD_Risk_Score,Clinical,numeric,90,10 to 99
2,Cholesterol_Level,Clinical,categorical,3,"Normal, High, Low"
3,Chronic_Kidney_Disease,Clinical,categorical,2,"No, Yes"
4,Diabetes,Clinical,categorical,2,"No, Yes"
5,Family_History_CVD,Clinical,categorical,2,"No, Yes"
6,Hypertension,Clinical,categorical,2,"No, Yes"
7,Obesity,Clinical,categorical,2,"No, Yes"
8,Previous_Heart_Attack,Clinical,categorical,2,"No, Yes"
9,Age,Demographic,numeric,60,30 to 89


The same columns, grouped:

In [8]:
pd.set_option("display.max_colwidth", None)
pd.DataFrame({"Group": list(GROUPS), "Columns": [", ".join(cols) for cols in GROUPS.values()]})

,Group,Columns
0,Identifier,Patient_ID
1,Demographic,"Age, Gender"
2,Clinical,"Hypertension, Diabetes, Obesity, Cholesterol_Level, Blood_Pressure, Chronic_Kidney_Disease, Previous_Heart_Attack, Family_History_CVD, CVD_Risk_Score, TCM_Use"
3,Lifestyle,"Smoking_Status, Alcohol_Consumption, Physical_Activity, Diet_Score, Stress_Level"
4,Environment,Air_Pollution_Exposure
5,Geography,"Region, Province, Rural_or_Urban"
6,Healthcare access,"Healthcare_Access, Hospital_Availability"
7,Socioeconomic,"Employment_Status, Education_Level, Income_Level"
8,Outcome,"Heart_Attack, Heart_Attack_Flag"


## 3. Data quality checks

In [9]:
num_cols = ["Age", "Blood_Pressure", "CVD_Risk_Score"]
cat_cols = df.select_dtypes(exclude="number").columns

print("Rows, columns:", df.shape)
print("Null values:", df.isnull().sum().sum())
print("Duplicate rows:", df.duplicated().sum())
print("Patient_ID unique:", df["Patient_ID"].is_unique)

# same category written two ways (e.g. "Yes" and "yes")?
print("Columns with spelling/case variants:", [c for c in cat_cols if df[c].str.strip().str.lower().nunique() != df[c].nunique()])

df[num_cols].describe()

Rows, columns: (239266, 28)
Null values: 0
Duplicate rows: 0
Patient_ID unique: True
Columns with spelling/case variants: []


,Age,Blood_Pressure,CVD_Risk_Score
count,239266.000000,239266.000000,239266.000000
mean,59.466284,134.513378,54.485581
std,17.348308,25.991827,25.987743
min,30.000000,90.000000,10.000000
25%,44.000000,112.000000,32.000000
50%,59.000000,135.000000,54.000000
75%,75.000000,157.000000,77.000000
max,89.000000,179.000000,99.000000


No nulls, no duplicate rows, `Patient_ID` is unique, and no category is written in two different ways. The numeric ranges are sensible (Age 30 to 89, Blood_Pressure 90 to 179, CVD_Risk_Score 10 to 99), so there is nothing to clean or remove.

## 4. Distributions

Numeric columns: KDE curves (hover to read the density).

In [14]:
for col in num_cols:
    values = df[col].dropna().to_numpy()
    x = np.linspace(values.min(), values.max(), 300)
    y = gaussian_kde(values)(x)
    fig = go.Figure()
    fig.add_trace(go.Histogram(
        x=values, histnorm="probability density", nbinsx=30,
        name="Histogram", opacity=0.45, marker_color="#7FA6C9"
    ))
    fig.add_trace(go.Scatter(
        x=x, y=y, mode="lines", name="KDE",
        line=dict(color="#D1495B", width=2.5)
    ))
    fig.update_layout(
        title=f"Distribution of {col}", xaxis_title=col,
        yaxis_title="Probability density", height=350,
        bargap=0.05, showlegend=True
    )
    fig.show()

Categorical columns: share of patients in each class (hover to read the exact percentage).

In [11]:
grid_cols = [c for c in cat_cols if c != "Province"]

counts = (df[grid_cols].melt(var_name="column", value_name="value")
            .groupby(["column", "value"]).size().reset_index(name="patients"))
counts["percent"] = counts["patients"] / len(df) * 100

# left-to-right order of the classes, so Low/Medium/High etc. do not get shuffled alphabetically
order = ["No", "Yes", "Female", "Male", "Non-Smoker", "Smoker", "Rural", "Urban", "Low", "Middle", "Medium", "Normal",
         "High", "Poor", "Moderate", "Good", "Healthy", "Employed", "Unemployed", "Retired", "None", "Primary",
         "Secondary", "Higher", "Central", "Eastern", "Northern", "Southern", "Western"]
counts["rank"] = counts["value"].map(order.index)
counts = counts.sort_values(["column", "rank"])

fig = px.bar(counts, x="value", y="percent", facet_col="column", facet_col_wrap=4,
             hover_data={"patients": ":,", "percent": ":.1f"},
             facet_col_spacing=0.06, facet_row_spacing=0.05, height=1700)
fig.update_xaxes(matches=None, showticklabels=True, title=None)
fig.update_yaxes(title=None)
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.update_layout(title="Share of patients in each class (%)")
fig.show()

In [12]:
province = df["Province"].value_counts(normalize=True).mul(100).round(1).sort_values()
px.bar(province, orientation="h", labels={"value": "% of patients", "Province": ""},
       title="Patients by province (%)", height=350).update_layout(showlegend=False)

The three numeric columns are almost flat: every age from 30 to 89, and every blood pressure and risk score value, is about equally common (the curves dip at the two ends only because the KDE smooths past the minimum and maximum). The categorical columns also split in very even shares. Gender and smoking are about 50/50, most three-level columns (cholesterol, activity, diet, stress, income, healthcare access, hospital availability, employment) are about a third each, education is about 25% in each class, the five regions are about 20% each and the eight provinces about 12.5% each. Air pollution is the exception at roughly 30% low, 50% medium and 20% high. `Heart_Attack` is imbalanced, with about 12% Yes.